# 03 · Whisper ASR

- Model sizes: `tiny / base / small` — chosen by `configs/config.yaml` and auto-downgraded on CPU/low VRAM.
- Supports `wav / mp3 / m4a / flac / ogg`; ffmpeg comes from `imageio-ffmpeg` if not installed system-wide.
- Output: `outputs/transcripts/<id>.json` (segments + timestamps) and `.txt`.
- **No speaker diarization**: speaker = `Unknown` for raw audio (documented future work).
- WER is computed only when a reference transcript exists (`jiwer`).

In [ ]:
print('[1/5] Hardware + model selection')
import sys; sys.path.insert(0, '.')
from src.config import load_config
from src.utils import get_hardware_info, select_models
cfg = load_config(); hw = get_hardware_info(); sel = select_models(cfg, hw)
print(hw); print('selected:', sel)

In [ ]:
print('[2/5] Provide audio')
from pathlib import Path
from src.config import resolve_path
sample = Path('data/samples/sample_meeting.wav')
if sample.exists():
    audio = sample
else:
    audio = None
    try:  # Colab upload
        from google.colab import files
        up = files.upload()
        audio = Path(list(up)[0])
    except Exception:
        print('No audio found. Either place a file at data/samples/sample_meeting.wav')
        print('or upload one (Colab). You can also skip to transcript-only mode in 09_end_to_end_demo.')
print('audio:', audio)

In [ ]:
print('[3/5] Transcribe (resumable: existing transcripts are skipped)')
from src.asr import WhisperASR
from src.utils import resolve_path as _rp
if audio is None:
    print('skipped: no audio')
else:
    asr = WhisperASR(model_size=sel['whisper'], device=sel['device'], language=cfg.whisper.language)
    result = asr.transcribe(audio)
    jp, tp = asr.save_transcript(result, resolve_path(cfg, 'transcripts'))
    print('segments:', len(result['segments']), '->', jp)
    for s in result['segments'][:5]:
        print(f"[{s['start']:7.1f}-{s['end']:7.1f}] {s['text']}")

In [ ]:
print('[4/5] Optional WER (only with a reference transcript)')
from src.asr import word_error_rate
reference = None   # set to 'the true transcript text' to enable
if reference and audio is not None:
    print('WER =', word_error_rate(reference, result['text']))
else:
    print('No reference provided -> WER: not evaluated (never fabricated).')

In [ ]:
print('[5/5] Saved artifacts')
from pathlib import Path
for p in sorted(Path('outputs/transcripts').glob('*')):
    print(f'  {p.name:40s} {p.stat().st_size:>10,d} B')